In [ ]:
# Install the allenai fork of lerobot that contains MolmoAct2 policy support.
!pip install -q 'git+https://github.com/allenai/lerobot.git@molmoact2-policy#egg=lerobot[molmoact2]'

# PyAV >= 12.0 required for av.option.Option used in lerobot's pyav_utils.py
!pip install -q 'av>=12.0.0' --upgrade

DEPRECATION: git+https://github.com/allenai/lerobot.git@molmoact2-policy#egg=lerobot[molmoact2] contains an egg fragment with a non-PEP 508 name pip 25.0 will enforce this behaviour change. A possible replacement is to use the req @ url syntax, and remove the egg fragment. Discussion can be found at https://github.com/pypa/pip/issues/11617
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 156.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 80.7 MB/s eta 0:00:00


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Patch pyav_utils.py so that the `av.option.Option` return-type annotation
# is not evaluated at import time (same trick as Pi0.5's vision-tower fix).
# This is needed because lerobot may pin av to a version that lacks av.option.
import glob

for path in glob.glob("/usr/local/lib/python3.*/dist-packages/lerobot/datasets/pyav_utils.py"):
    src = open(path).read()
    if "from __future__ import annotations" not in src:
        open(path, "w").write("from __future__ import annotations\n" + src)
        print(f"Patched: {path}")
    else:
        print(f"Already patched: {path}")

Patched: /usr/local/lib/python3.13/dist-packages/lerobot/datasets/pyav_utils.py


In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("Logged in to HuggingFace.")

Logged in to HuggingFace.


In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_REPO_ID = "subhodipsaha/so101_stack_three_cubes_08_19"

# ── Pretrained checkpoint ─────────────────────────────────────────────────────
CHECKPOINT_PATH = "allenai/MolmoAct2-SO100_101"

# ── Fine-tuned model repo ID (your HF username/model-name) ───────────────────
POLICY_REPO_ID  = "subhodipsaha/molmoact2_stack_three_cubes_v2"

# ── Training mode ─────────────────────────────────────────────────────────────
TRAIN_MODE_VLM = "freeze"   # action expert only — VLM frozen

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/content/drive/MyDrive/molmoact2_stack_three_cubes_v2"

# ── Training hyperparams ──────────────────────────────────────────────────────
# Dataset: 119 episodes, 74,399 frames
# Epochs = (BATCH_SIZE * STEPS) / 74399
#   bs=32, steps=2500 → 80,000 samples → 1.07 epochs  (~9h on Colab L4)
#   bs=32, steps=3500 → 112,000 samples → 1.51 epochs  (resume if loss still falling)
# NOTE: bs=32 may OOM on L4 (~15GB at bs=16). Fall back to bs=16 if it does.
BATCH_SIZE  = 32
STEPS       = 2500
LOG_FREQ    = 100
SAVE_FREQ   = 500

epochs = (BATCH_SIZE * STEPS) / 74399
print(f"Dataset    : {DATASET_REPO_ID}")
print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Policy repo: {POLICY_REPO_ID}")
print(f"Train mode : {TRAIN_MODE_VLM}")
print(f"Batch size : {BATCH_SIZE}  |  Steps: {STEPS}  |  Epochs: {epochs:.2f}")
print(f"Output     : {OUTPUT_DIR}")

In [ ]:
import shutil, pathlib

hf_hub_name = DATASET_REPO_ID.replace("/", "--")
paths_to_clear = [
    # allenai lerobot fork cache (confirmed location)
    pathlib.Path(f"/root/.cache/huggingface/lerobot/hub/datasets--{hf_hub_name}"),
    # standard hub cache
    pathlib.Path(f"/root/.cache/huggingface/hub/datasets--{hf_hub_name}"),
    # lerobot own cache
    pathlib.Path(f"/root/.cache/huggingface/lerobot/{DATASET_REPO_ID}"),
    # HF datasets library arrow cache (all parquet entries)
    pathlib.Path("/root/.cache/huggingface/datasets/parquet"),
]
for p in paths_to_clear:
    if p.exists():
        shutil.rmtree(p)
        print(f"Cleared: {p}")
    else:
        print(f"Not found (ok): {p}")

In [ ]:
import subprocess
import sys
import shutil
from pathlib import Path

# Delete leftover empty output dir so lerobot can create it fresh
ckpt_dirs = list(Path(OUTPUT_DIR).glob("checkpoints/*/pretrained_model")) if Path(OUTPUT_DIR).exists() else []
if Path(OUTPUT_DIR).exists() and not ckpt_dirs:
    shutil.rmtree(OUTPUT_DIR)
    print(f"Removed leftover output dir: {OUTPUT_DIR}")

# Pin to the exact current commit so Colab cannot resolve to an old snapshot
DATASET_REVISION = "8311b76246ef0a02f33f996edbef40031251a85b"

cmd = [
    sys.executable, "-m", "lerobot.scripts.lerobot_train",
    f"--dataset.repo_id={DATASET_REPO_ID}",
    f"--dataset.revision={DATASET_REVISION}",
    "--dataset.video_backend=pyav",
    "--policy.type=molmoact2",
    f"--policy.repo_id={POLICY_REPO_ID}",
    f"--policy.checkpoint_path={CHECKPOINT_PATH}",
    f"--policy.train_mode_vlm={TRAIN_MODE_VLM}",
    "--policy.action_mode=continuous",
    "--policy.model_dtype=bfloat16",
    "--policy.gradient_checkpointing=true",
    "--policy.normalize_gripper=true",  # gripper values in dataset are outside [-1,1]
    f"--batch_size={BATCH_SIZE}",
    f"--steps={STEPS}",
    f"--log_freq={LOG_FREQ}",
    f"--save_freq={SAVE_FREQ}",
    f"--output_dir={OUTPUT_DIR}",
    "--wandb.enable=false",
]

print("Running:", " ".join(cmd), flush=True)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"lerobot-train exited with code {process.returncode}")

In [ ]:
print("hi")